# 05. Descriptive stats

Things to do in this section (modelling from the sql file):
- per symbol stats: how did each individual stock behave?
- per sector stats but all days and stocks combined: what does an individual stock-day within each sector tend to look like? Every stock's influence on the sector stat is proportional to how many days of data it contributes. Ignores time and diversification. Treats every (stock+day) return as one independent observation and pools them all within a sector.
- per sector stats but also per day: how did an equal weighted sector return series behave through time? Every avg(simple_return) is computed within each date over whichever symbols are active that day, so that it returns an equal-weighted avg of that day's actual constituents. On any given day, no stock dominates the number. First it collapses all stocks into a single equal-weighted daily return per sector (date+sector), building an implicit sector index, then runs the same stats on that one time series per sector. Builds a daily sector portfolio first, so diversifying per day.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from pathlib import Path

source= Path(r"filepath\csv_features")
companies_source= Path(r"filepath\csv_clean")

stocks=pd.read_csv(source/"stock_features.csv", parse_dates=["date"], date_format="%Y-%m-%d")
companies=pd.read_csv(companies_source/"companies_clean.csv")

#### 05.1 Per symbol descriptive stats

Basic stats:

In [ ]:
# the main metrics used to get all stats will be simple_return without NaNs
non_nan_returns=stocks.dropna(subset=["simple_return"])

# group the data by stock:
grouped=non_nan_returns.groupby("symbol")["simple_return"]

stock_stats=grouped.agg(
    n_obs="count", 
    mean_return="mean", 
    median_return="median", 
    sample_std_return="std", 
    pop_std_return=lambda x: x.std(ddof=0),
    min_return="min", 
    max_return="max",
    skewness=lambda x: skew(x, bias=True),
    #bias=True is to allow bias for the population. do not correct to N-1, to mimic sql track
    excess_kurtosis=lambda x: kurtosis(x,bias=True)
    #the fisher=True is default. what actually calcs the excess kurtosis and not just the kurtosis                     
).reset_index()

stock_stats["annualized_mean_return"]=stock_stats["mean_return"] * 252
stock_stats["annualized_volatility"]=stock_stats["sample_std_return"] * np.sqrt(252)
stock_stats["sharpe_ratio_naive"]=stock_stats["annualized_mean_return"]/stock_stats["annualized_volatility"]

stock_stats

,symbol,n_obs,mean_return,median_return,sample_std_return,pop_std_return,min_return,max_return,skewness,excess_kurtosis,annualized_mean_return,annualized_volatility,sharpe_ratio_naive
0,A,3767,0.000668,0.000895,0.017986,0.017983,-0.110139,0.117631,-0.196235,4.552871,0.168218,0.285515,0.589174
1,AAPL,3767,0.001130,0.001001,0.017561,0.017559,-0.128647,0.119808,-0.043645,5.187642,0.284876,0.278780,1.021866
2,ABBV,3013,0.000834,0.001306,0.016608,0.016605,-0.162524,0.137673,-0.676224,10.770506,0.210241,0.263639,0.797459
3,ABNB,1013,0.000403,0.000000,0.030936,0.030920,-0.134250,0.133532,0.123672,2.160297,0.101640,0.491088,0.206969
4,ABT,3767,0.000571,0.000594,0.013512,0.013510,-0.097857,0.109360,-0.142310,7.077064,0.143899,0.214497,0.670866
...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,XYL,3318,0.000666,0.000772,0.016583,0.016580,-0.120273,0.124438,-0.007153,7.318731,0.167823,0.263246,0.637514
498,YUM,3767,0.000628,0.000805,0.014894,0.014892,-0.188324,0.232484,0.261436,27.411868,0.158257,0.236430,0.669363
499,ZBH,3767,0.000321,0.000186,0.016172,0.016169,-0.139943,0.155798,0.159103,10.005812,0.080826,0.256716,0.314847
500,ZBRA,3767,0.000951,0.001032,0.022539,0.022536,-0.238459,0.172142,-0.290026,11.908642,0.239662,0.357799,0.669821


final_cumulative_return and annualized_compound_return

In [ ]:
final_cum_ret=(
    stocks
    .sort_values(["symbol","date"])
    .groupby("symbol").last()["cumulative_return"]
    .rename("final_cumulative_return")
    .reset_index()
)

"""
	symbol	final_cumulative_return
0	A	5.707714
1	AAPL	38.471658
2	ABBV	7.117740
...

"""
#let's merge this with stock_stats table by the symbol:
stock_stats=stock_stats.merge(final_cum_ret,on="symbol")


#get the annualized compound return:
stock_stats["annualized_compound_return"]=(
    (1+stock_stats["final_cumulative_return"]) ** (252 / stock_stats["n_obs"])-1
)


"\nThe cumulative_return is the total rate for my return. The amount my investment grew for the entire period.\n In this dataset, the cumulative return is just: current_adj_close/first_adj_close - 1\nThe annualized_compound_return is the yearly rate i need in order to get that total rate. It converts to \n a constant yearly rate that my investment needs to suffer to get to the overall value.\nSo i have the final cumulative return (form the start to end period my investment grew final_cumul_return %),\n this is my r (because i dont care how many years, im just getting the final value of my investment Vf=Vi(1+r)),\n and with the above formula (annualized_compound_return) i want to know what should'v been my annual return R,\n to get that final value. So, (1+r)=(1+R)^n, and solving to R. So i know that the period of my stock is 10 years\n and i want to know the yearly rate. \n\nSo I know that my investment grew by 60% over the entire period (from start to finish). \nI want to find the ann

max_drawdown:

In [ ]:
#For this i need to make sure my data is ordered by symbol and date.
# Then i need to group by symbol and to get the maximum value so far (cummax()).
stocks_sorted = stocks.sort_values(["symbol", "date"]).copy()
running_max = stocks_sorted.groupby("symbol")["adj_close"].cummax()

stocks_sorted["drawdown"] = (stocks_sorted["adj_close"] / running_max) - 1

max_drawdown=stocks_sorted.groupby("symbol")["drawdown"].min().rename("max_drawdown").reset_index()

stock_stats = stock_stats.merge(max_drawdown, on="symbol")
stock_stats.head()

,symbol,n_obs,mean_return,median_return,sample_std_return,pop_std_return,min_return,max_return,skewness,excess_kurtosis,annualized_mean_return,annualized_volatility,sharpe_ratio_naive,final_cumulative_return,annualized_compound_return,max_drawdown
0,A,3767,0.000668,0.000895,0.017986,0.017983,-0.110139,0.117631,-0.196235,4.552871,0.168218,0.285515,0.589174,5.707714,0.135782,-0.440852
1,AAPL,3767,0.001130,0.001001,0.017561,0.017559,-0.128647,0.119808,-0.043645,5.187642,0.284876,0.278780,1.021866,38.471658,0.278752,-0.437971
2,ABBV,3013,0.000834,0.001306,0.016608,0.016605,-0.162524,0.137673,-0.676224,10.770506,0.210241,0.263639,0.797459,7.117740,0.191415,-0.450898
3,ABNB,1013,0.000403,0.000000,0.030936,0.030920,-0.134250,0.133532,0.123672,2.160297,0.101640,0.491088,0.206969,-0.072559,-0.018564,-0.619581
4,ABT,3767,0.000571,0.000594,0.013512,0.013510,-0.097857,0.109360,-0.142310,7.077064,0.143899,0.214497,0.670866,5.087815,0.128439,-0.338768


Stock stats summary:

In [ ]:
summary_cols = ["annualized_mean_return", "annualized_volatility", "sharpe_ratio_naive",
                "annualized_compound_return", "max_drawdown", "skewness", "excess_kurtosis"]
stock_stats[summary_cols].describe().round(2).T.style.format({
    "count": "{:.0f}",
    "mean": "{:.2%}",
    "std": "{:.2%}",
    "min": "{:.2%}",
    "25%": "{:.2%}",
    "50%": "{:.2%}",
    "75%": "{:.2%}",
    "max": "{:.2%}"
})



'Ou\n\nstock_stats[["annualized_volatility", "max_drawdown"]].agg("median")\n\nOU \n\nstock_stats.agg({\n    "annualized_volatility": "median",\n    "max_drawdown": "median"\n})\n\nOU\n\nstock_stats.agg({\n    "annualized_volatility": ["median", "mean"],\n    "max_drawdown": ["median", "mean"]\n})\n'

In [8]:
#To fix the %. Naive, Kurt and Skew are not %. 

cols = ["annualized_mean_return", "annualized_volatility",
        "annualized_compound_return", "max_drawdown",
        "sharpe_ratio_naive", "skewness", "excess_kurtosis"]

summary = stock_stats[cols].describe().T
summary.loc[cols[:4], "mean":] *= 100   # turn the return columns into % values
summary = summary.rename(index=lambda c: c + " (%)" if c in cols[:4] else c)
summary.round(2)

,count,mean,std,min,25%,50%,75%,max
annualized_mean_return (%),502.0,18.10,12.32,-135.07,12.79,16.73,21.80,132.58
annualized_volatility (%),502.0,30.77,9.06,16.71,24.78,28.63,34.79,90.30
annualized_compound_return (%),502.0,14.59,14.13,-82.68,9.09,13.32,18.27,234.45
max_drawdown (%),502.0,-53.64,15.70,-96.47,-63.59,-51.29,-43.02,-13.43
sharpe_ratio_naive,502.0,0.60,0.26,-1.50,0.46,0.60,0.74,2.74
skewness,502.0,-0.03,0.90,-6.37,-0.28,-0.03,0.18,11.35
excess_kurtosis,502.0,15.13,23.88,0.17,7.63,10.66,15.69,395.97


#### 05.2 Cross sectional (Per sector)

Every stock and day pooled as one observation. "What does a random stock-day within this sector look like?"
Don't care about time series. Only the sector as a whole (all stocks and days of that sector).
Each stock contributes differently to the sector. A stock with more daily observations will obviously have a heavier weight on the sector.
Sum all the values of every stock per sector than divide by the amount of stocks in that sector.

Getting the base tables:

In [ ]:
#Already have the stocks table wihtou the first nulls: non_nan_returns
#Need to merge with companies to get the sector
sector_returns= non_nan_returns.merge(companies[["symbol","sector"]],on="symbol")

#Now it's time to group by sector, but again my calculation will only be on simple_return:
sector_grouped=sector_returns.groupby("sector")["simple_return"]



Basic stats per sector:

In [ ]:
#Now let's get the stats:
sector_stats=sector_grouped.agg(
    n_obs="count",
    mean_return="mean",
    median_return="median",
    sample_std_return="std",
    pop_std_return=lambda x: x.std(ddof=0),
    min_return="min",
    max_return="max",
    skewness=lambda x: skew(x,bias=True),
    excess_kurtosis=lambda x: kurtosis(x,bias=True, fisher=True)
    #bias=True to divide by N and not by N-1. To mimic our sql.
    #Fisher=True to get the excess kurtosis and not just the kurtosis.
)

sector_stats["n_symbols"]=sector_returns.groupby("sector")["symbol"].nunique()

sector_stats=sector_stats.reset_index()

sector_stats

,sector,n_obs,mean_return,median_return,sample_std_return,pop_std_return,min_return,max_return,skewness,excess_kurtosis,n_symbols
0,Basic Materials,78115,0.000570,0.000546,0.020827,0.020827,-0.275077,0.296846,-0.058716,9.327740,22
1,Communication Services,75911,0.000702,0.000590,0.020769,0.020768,-0.365884,0.422235,0.157041,19.678769,22
2,Consumer Cyclical,196421,0.000812,0.000616,0.022086,0.022085,-0.375050,0.441571,0.084095,18.619560,55
3,Consumer Defensive,132912,0.000500,0.000636,0.014896,0.014896,-0.321463,0.372334,-0.189228,26.454997,37
4,Energy,80714,0.000636,0.000500,0.024684,0.024683,-0.538647,0.373199,-0.333706,25.325373,22
5,Financial Services,248420,0.000686,0.000836,0.018320,0.018320,-0.531402,0.276952,-0.150291,15.576047,67
6,Healthcare,221793,0.000684,0.000675,0.018958,0.018958,-0.406583,0.619066,0.633917,34.287996,62
7,Industrials,246056,0.000770,0.000804,0.018836,0.018836,-0.302902,0.676812,0.307631,19.851589,70
8,Real Estate,112982,0.000555,0.000872,0.017451,0.017450,-0.618478,0.349780,-0.462112,35.448446,31
9,Technology,285007,0.000922,0.000953,0.022574,0.022574,-0.456763,0.522901,0.255399,17.979667,82


#### 05.3 Sector performance (Per sector+day)

Avg return across all symbols active in the sector each day. Time ordered sector performance.
Each stock will have the same daily contribution to the sector.
Sum all the values of every stock of a sector for that day and divide by the number of those stocks. So on each date, Technology, for ex., becomes one single number.
Now i have a proper sector-level time series, in order to calculate metrics that actually describe the performance of an equal-weighted sector portfolio:
- cumulative return
- annualized return
- volatility
- drawdown
- Sharpe-style metric
The previous pooled sector data is not a time series of one portfolio. It is a collection of many stock-day observations.

Get the base tables:

In [ ]:
sector_daily=(
    sector_returns.groupby(["sector","date"])["simple_return"]
    .mean()
    .rename("sector_return")
    .reset_index()
    #.dropna(subset=["sector_return"]) not needed because it's based on non_nan_returns
    .sort_values(["sector","date"])
)

sector_daily["cum_index"]=sector_daily.groupby("sector")["sector_return"].transform(lambda x: (1+x).cumprod())


Get the stats:

In [ ]:
sector_date_stats=sector_daily.groupby("sector")["sector_return"].agg(
    n_obs="count",
    mean_return="mean",
    median_return="median",
    sample_std_return="std",
    pop_std_return=lambda x: x.std(ddof=0),
    min_return="min",
    max_return="max",
    skewness=lambda x: skew(x, bias=True),
    excess_kurtosis=lambda x: kurtosis(x, bias=True, fisher=True)
    #bias=True to divide by N and not by N-1. To mimic our sql.
    #Fisher=True to get the excess kurtosis and not just the kurtosis.
)


'\nOr for skewness and kurt:\n\nsector_perf_sk = sg.apply(\n    lambda s: pd.Series({\n        "skewness": skew(s, bias=True),\n        "excess_kurtosis": kurtosis(s, bias=True, fisher=True),\n    })\n).reset_index()\n\nsector_perf = sector_perf.merge(sector_perf_sk, on="sector")\n'

annualized mean return, annualized volatility, sharpe ratio naive and annualized compound return:

In [ ]:
#annualized_mean_return
sector_date_stats["annualized_mean_return"]=sector_date_stats["mean_return"]*252
#annualized_volatility
sector_date_stats["annualized_volatility"]=sector_date_stats["sample_std_return"]*np.sqrt(252)
#sharpe_ratio_naive
sector_date_stats["sharpe_ratio_naive"]=sector_date_stats["annualized_mean_return"]/sector_date_stats["annualized_volatility"]

#final_cum_growth:
final_cum_growth=sector_daily.groupby("sector").last()["cum_index"].rename("final_cum_growth").reset_index()
#let's merge the new column with the df on sector:
sector_date_stats=sector_date_stats.merge(final_cum_growth,on="sector")

#converting the growth into a return:
sector_date_stats["final_cum_ret"]=sector_date_stats["final_cum_growth"]-1

#annualized compound return:
sector_date_stats["annualized_compound_return"]=(sector_date_stats["final_cum_growth"])**(252/sector_date_stats["n_obs"])-1




max drawdown:

In [ ]:
sector_running_max = sector_daily.groupby("sector")["cum_index"].cummax()

sector_daily["drawdown"] = (sector_daily["cum_index"] / sector_running_max) - 1

sector_max_drawdown=sector_daily.groupby("sector")["drawdown"].min().rename("sector_max_drawdown").reset_index()

sector_date_stats = sector_date_stats.merge(sector_max_drawdown, on="sector")
sector_date_stats.sort_values("sharpe_ratio_naive", ascending=False)

,sector,n_obs,mean_return,median_return,sample_std_return,pop_std_return,min_return,max_return,skewness,excess_kurtosis,annualized_mean_return,annualized_volatility,sharpe_ratio_naive,final_cum_growth,final_cum_ret,annualized_compound_return,sector_max_drawdown
9,Technology,3767,0.000914,0.001323,0.014085,0.014083,-0.132808,0.113947,-0.308670,6.706226,0.230295,0.223585,1.030009,21.470217,20.470217,0.227709,-0.343852
6,Healthcare,3767,0.000688,0.000981,0.010979,0.010978,-0.103518,0.081605,-0.372875,6.831508,0.173436,0.174287,0.995118,10.636371,9.636371,0.171357,-0.294703
2,Consumer Cyclical,3767,0.000821,0.001093,0.013612,0.013610,-0.153067,0.161050,-0.315953,16.528645,0.206787,0.216079,0.956999,15.485971,14.485971,0.201166,-0.486566
7,Industrials,3767,0.000767,0.001151,0.012770,0.012769,-0.116493,0.126451,-0.308414,9.266705,0.193404,0.202723,0.954032,13.226818,12.226818,0.188562,-0.405817
1,Communication Services,3767,0.000716,0.001102,0.012207,0.012206,-0.097173,0.083076,-0.409923,5.722380,0.180452,0.193788,0.931182,11.191721,10.191721,0.175352,-0.373540
3,Consumer Defensive,3767,0.000506,0.000843,0.008658,0.008657,-0.093326,0.082501,-0.409428,11.438659,0.127590,0.137443,0.928313,5.843704,4.843704,0.125353,-0.242144
5,Financial Services,3767,0.000687,0.001138,0.013661,0.013659,-0.144786,0.130103,-0.287093,12.986452,0.173064,0.216861,0.798044,9.335021,8.335021,0.161175,-0.435175
10,Utilities,3767,0.000522,0.000932,0.011244,0.011243,-0.123804,0.115928,-0.252739,15.148568,0.131474,0.178493,0.736579,5.619689,4.619689,0.122415,-0.371001
8,Real Estate,3767,0.000558,0.000975,0.013077,0.013076,-0.180855,0.092316,-0.693600,16.877215,0.140676,0.207597,0.677638,5.918517,4.918517,0.126312,-0.418173
0,Basic Materials,3767,0.000569,0.000843,0.014327,0.014325,-0.132270,0.120603,-0.375659,7.337102,0.143451,0.227427,0.630756,5.788679,4.788679,0.124641,-0.415407


#### 05.4 Rankings

Stock stats ranking:

by final cumulative return:

In [ ]:
cols = ["symbol", "n_obs", "final_cumulative_return", "annualized_compound_return", "sharpe_ratio_naive", "max_drawdown"]
print("Top 10 by TOTAL return over available history:")
print(stock_stats.nlargest(10, "final_cumulative_return")[cols])

print()
print("Bottom 10 by total return:")
print(stock_stats.nsmallest(10, "final_cumulative_return")[cols])

Top 10 by TOTAL return over available history:
    symbol  n_obs  final_cumulative_return  annualized_compound_return  \
44    AVGO   3767              1743.416299                    0.647615   
344   NVDA   3767               316.746655                    0.470216   
450   TSLA   3645               263.374161                    0.470490   
443    TPL   3767               155.754769                    0.402339   
47    AXON   3767               134.847310                    0.388974   
334   NFLX   3767               117.985603                    0.376714   
187   FICO   3767                96.844169                    0.358815   
465    URI   3767                72.519178                    0.333080   
143    DPZ   3767                59.556389                    0.315893   
287   LRCX   3767                55.653108                    0.310041   

     sharpe_ratio_naive  max_drawdown  
44             1.484695     -0.483000  
344            1.073525     -0.663351  
450            0.9

by annualized compound return:

In [16]:
cols3 = ["symbol", "n_obs", "annualized_compound_return", "final_cumulative_return", "max_drawdown"]
print("Top 10 by ANNUALIZED compound return (fair cross-stock comparison):")
print(stock_stats.nlargest(10, "annualized_compound_return")[cols3])
print()
print("Bottom 10 by annualized compound return:")
print(stock_stats.nsmallest(10, "annualized_compound_return")[cols3])

Top 10 by ANNUALIZED compound return (fair cross-stock comparison):
    symbol  n_obs  annualized_compound_return  final_cumulative_return  \
202    GEV    184                    2.344450                 1.414573   
86     CEG    735                    0.799695                 4.550479   
374   PLTR   1063                    0.659882                 7.478948   
44    AVGO   3767                    0.647615              1743.416299   
450   TSLA   3645                    0.470490               263.374161   
344   NVDA   3767                    0.470216               316.746655   
76    CARR   1198                    0.448490                 4.820863   
443    TPL   3767                    0.402339               155.754769   
114   CRWD   1392                    0.393273                 5.246379   
33    ANET   2654                    0.393015                31.817454   

     max_drawdown  
202     -0.136037  
86      -0.276450  
374     -0.846154  
44      -0.483000  
450     -0.736322

by sharpe ratio naive:

In [17]:
cols2 = ["symbol", "sharpe_ratio_naive", "annualized_mean_return", "annualized_volatility", "max_drawdown"]
print("Top 10 by Sharpe ratio:")
print(stock_stats.nlargest(10, "sharpe_ratio_naive")[cols2])
print()
print("Bottom 10 by Sharpe ratio:")
print(stock_stats.nsmallest(10, "sharpe_ratio_naive")[cols2])

Top 10 by Sharpe ratio:
    symbol  sharpe_ratio_naive  annualized_mean_return  annualized_volatility  \
202    GEV            2.739208                1.325753               0.483991   
86     CEG            1.555022                0.682546               0.438930   
44    AVGO            1.484695                0.573669               0.386389   
118   CTAS            1.188194                0.298199               0.250968   
470   VLTO            1.164112                0.271323               0.233073   
76    CARR            1.160865                0.441731               0.380519   
107   COST            1.132618                0.228850               0.202054   
344   NVDA            1.073525                0.487989               0.454567   
187   FICO            1.068788                0.363828               0.340412   
18     AJG            1.065861                0.216933               0.203528   

     max_drawdown  
202     -0.136037  
86      -0.276450  
44      -0.483000  
118 

#### 05.5 Sanity checks:

No positive drawdowns:

In [18]:
print("n_symbols (expect 502):", len(stock_stats))
print("n_sectors (expect 11):", len(sector_date_stats))
print("Positive drawdowns, stocks:", (stock_stats["max_drawdown"] > 0).sum())
print("Positive drawdowns, sectors:", (sector_date_stats["sector_max_drawdown"] > 0).sum())

n_symbols (expect 502): 502
n_sectors (expect 11): 11
Positive drawdowns, stocks: 0
Positive drawdowns, sectors: 0


No negative standard deviations:

In [19]:
#negative std deviations?
print((
    (stock_stats["sample_std_return"] < 0) |
    (stock_stats["annualized_volatility"] < 0)
).sum())
#0

#any nulls? check if there are nulls in any column (that's why axis=1, to check columns)
print(stock_stats[stock_stats.isnull().any(axis=1)])
#empty df

0
Empty DataFrame
Columns: [symbol, n_obs, mean_return, median_return, sample_std_return, pop_std_return, min_return, max_return, skewness, excess_kurtosis, annualized_mean_return, annualized_volatility, sharpe_ratio_naive, final_cumulative_return, annualized_compound_return, max_drawdown]
Index: []


How many tickers actually contribute to each sector's equal-weighted daily return?

In [20]:

coverage = (
    sector_returns.groupby(["sector", "date"])["symbol"].count().reset_index(name="n")
    .groupby("sector")["n"].agg(min_tickers_per_day="min", max_tickers_per_day="max", avg_tickers_per_day="mean")
    .reset_index()
)
coverage["range"] = coverage["max_tickers_per_day"] - coverage["min_tickers_per_day"]
coverage.sort_values("range", ascending=False)

,sector,min_tickers_per_day,max_tickers_per_day,avg_tickers_per_day,range
9,Technology,63,82,75.658880,19
7,Industrials,60,70,65.318821,10
2,Consumer Cyclical,46,55,52.142554,9
6,Healthcare,55,62,58.877887,7
1,Communication Services,16,22,20.151580,6
4,Energy,17,22,21.426599,5
5,Financial Services,62,67,65.946376,5
0,Basic Materials,19,22,20.736660,3
3,Consumer Defensive,34,37,35.283249,3
10,Utilities,29,32,29.792408,3


#### 05.6 Saving

In [ ]:
destination = Path(r"filepath\csv_stats")

stock_stats.to_csv(destination / "stock_descriptive_stats.csv", index=False)
sector_stats.to_csv(destination / "sector_stock_return_distribution.csv", index=False)
sector_date_stats.to_csv(destination / "sector_performance_stats.csv", index=False)
sector_daily[["date", "sector", "sector_return"]].to_csv(destination / "sector_daily_returns.csv", index=False)
